# 04 — Part 2: association of five maternal factors with infant death

**Goal.** Estimate, one factor at a time, how much pre-pregnancy hypertension, pre-pregnancy diabetes, pre-pregnancy BMI class, smoking before pregnancy and the interval since the last pregnancy change the risk of death before age one, after adjustment for confounders chosen from a DAG.

**Thesis part.** Part 2 (association). Plan: `plan/analysis_plan.md`, section 4. Methods and pass/fail rules were fixed before any result.

**Status.** Only the method is here so far: the model code, its check on synthetic data, and the DAG check. The analysis on the real data comes next.

**Inputs.** For now none (synthetic data only). Later: `data/parquet/cohort_<year>.parquet`.

**Outputs.** Later: `reports/part2_results.csv` and forest plots.

**How to run.** Run all cells (about 10 seconds).

**Terms.**
- *Risk ratio (RR)*: the risk of death with the factor divided by the risk without it.
- *Risk difference (RD)*: the extra deaths per 1,000 births caused by the factor, if the association is causal.
- *Confounder*: a cause of both the factor and death; we adjust for it.

## 1. Setup

In [1]:
import numpy as np
import polars as pl
import statsmodels.api as sm

## 2. Modified Poisson regression on covariate-pattern counts

Modified Poisson regression (log link, robust SE) fitted on covariate-pattern counts.

**Why counts.** every covariate in Part 2 is a category. Births with the same
categories share one predicted risk mu. So the model can be fitted on one row
per pattern, with n = births and d = deaths in that pattern, instead of on
millions of single births. Estimates are identical.

**Robust (sandwich, HC0) covariance, exact from counts:**

    bread  A = sum_g n_g * mu_g * x_g x_g'
    meat   B = sum_g [ d_g (1 - mu_g)^2 + (n_g - d_g) mu_g^2 ] x_g x_g'
    cov    = A^-1 B A^-1
(the bracket is the sum of squared single-birth residuals y - mu in the pattern).
Zou (2004) "modified Poisson" = Poisson log-link model + this robust covariance.

In [2]:
def design(rows: dict, columns: list[str], reference: dict) -> tuple[np.ndarray, list[str]]:
    """Intercept + one 0/1 column per non-reference category.

    rows: {column: numpy array of category codes (strings or ints), one per pattern}
    reference: {column: reference category}
    """
    n = len(next(iter(rows.values())))
    mats, names = [np.ones((n, 1))], ["intercept"]
    for col in columns:
        vals = rows[col]
        cats = [c for c in sorted(set(vals.tolist()), key=str) if c != reference[col]]
        for c in cats:
            mats.append((vals == c).astype(np.float64)[:, None])
            names.append(f"{col}={c}")
    return np.hstack(mats), names


def fit(X: np.ndarray, n: np.ndarray, d: np.ndarray, max_iter: int = 100, tol: float = 1e-10) -> dict:
    """Fit log mu = X beta on pattern counts (n births, d deaths). Returns beta and covariances."""
    n, d = n.astype(np.float64), d.astype(np.float64)
    beta = np.zeros(X.shape[1])
    beta[0] = np.log(d.sum() / n.sum())
    for it in range(max_iter):
        mu = np.exp(X @ beta)
        score = X.T @ (d - n * mu)
        A = (X * (n * mu)[:, None]).T @ X
        step = np.linalg.solve(A, score)
        beta += step
        if np.max(np.abs(step)) < tol:
            break
    else:
        raise RuntimeError("modified Poisson did not converge")
    mu = np.exp(X @ beta)
    A = (X * (n * mu)[:, None]).T @ X
    B = (X * (d * (1 - mu) ** 2 + (n - d) * mu ** 2)[:, None]).T @ X
    A_inv = np.linalg.inv(A)
    return {"beta": beta, "cov_robust": A_inv @ B @ A_inv, "cov_model": A_inv, "iterations": it + 1,
            "max_mu": float(mu.max())}


def rr(result: dict, names: list[str], term: str) -> dict:
    """Risk ratio, 95% CI and SE(log RR) for one coefficient."""
    j = names.index(term)
    b, se = result["beta"][j], np.sqrt(result["cov_robust"][j, j])
    return {"term": term, "log_rr": b, "se": se, "rr": np.exp(b),
            "rr_lo": np.exp(b - 1.96 * se), "rr_hi": np.exp(b + 1.96 * se)}


def standardized_rd(result: dict, X: np.ndarray, n: np.ndarray, exposed: np.ndarray,
                    exposure_cols: list[int], draws: int = 1000, seed: int = 20261123) -> dict:
    """g-computation risk difference per 1,000 births, standardized to the exposed patterns.

    For births in exposed patterns: predicted risk as observed minus predicted risk
    with the exposure columns set to 0 (reference). CI from coefficient draws
    from N(beta, robust covariance).
    """
    Xe, ne = X[exposed], n[exposed].astype(np.float64)
    X0 = Xe.copy()
    X0[:, exposure_cols] = 0.0

    def rd(beta):
        return 1000 * np.sum(ne * (np.exp(Xe @ beta) - np.exp(X0 @ beta))) / ne.sum()

    rng = np.random.default_rng(seed)
    sims = rng.multivariate_normal(result["beta"], result["cov_robust"], size=draws)
    vals = np.array([rd(b) for b in sims])
    return {"rd_per_1000": rd(result["beta"]), "rd_lo": np.percentile(vals, 2.5), "rd_hi": np.percentile(vals, 97.5)}

## 3. Check against statsmodels on synthetic data

**What.** We simulate 300,000 births with known effects, fit the same model (a) with statsmodels on single births and (b) with our function on pattern counts, and compare.

**Pass rule.** Coefficients, robust standard errors and the g-computation risk difference must agree to better than 1e-8. No real data is used.

In [3]:
rng = np.random.default_rng(1)
N = 300_000
age, race = rng.integers(1, 6, N), rng.integers(1, 4, N)
expo = (rng.random(N) < 0.03 + 0.01 * (age > 3)).astype(int)
y = (rng.random(N) < 0.004 * np.exp(0.6 * expo + 0.15 * (age - 3) + 0.3 * (race == 2))).astype(float)

Xi = np.column_stack([np.ones(N)] + [(age == a).astype(float) for a in range(2, 6)]
                     + [(race == r).astype(float) for r in (2, 3)] + [expo.astype(float)])
ref = sm.GLM(y, Xi, family=sm.families.Poisson()).fit(cov_type="HC0")

g = pl.DataFrame({"age": age, "race": race, "expo": expo, "y": y}).group_by("age", "race", "expo") \
      .agg(n=pl.len(), d=pl.col("y").sum())
X, names = design({c: g[c].to_numpy() for c in ("age", "race", "expo")}, ["age", "race", "expo"],
                  {"age": 1, "race": 1, "expo": 0})
res = fit(X, g["n"].to_numpy(), g["d"].to_numpy())
order = [names.index(t) for t in ["intercept", "age=2", "age=3", "age=4", "age=5", "race=2", "race=3", "expo=1"]]

beta_diff = np.max(np.abs(res["beta"][order] - ref.params))
se_diff = np.max(np.abs(np.sqrt(np.diag(res["cov_robust"]))[order] / ref.bse - 1))
exposed = g["expo"].to_numpy() == 1
rd_grouped = standardized_rd(res, X, g["n"].to_numpy(), exposed, [names.index("expo=1")], draws=100)["rd_per_1000"]
Xe = Xi[expo == 1]
X0 = Xe.copy()
X0[:, -1] = 0
rd_single = 1000 * np.mean(np.exp(Xe @ ref.params) - np.exp(X0 @ ref.params))

print(f"beta diff {beta_diff:.1e}, SE diff {se_diff:.1e}, RD {rd_grouped:.6f} vs {rd_single:.6f}, "
      f"RR {rr(res, names, 'expo=1')['rr']:.4f}")
assert beta_diff < 1e-8 and se_diff < 1e-8 and abs(rd_grouped - rd_single) < 1e-8
print("PASS")

beta diff 1.3e-12, SE diff 4.7e-12, RD 3.656052 vs 3.656052, RR 1.7947
PASS


**Result. PASS.** The two fits agree: the coefficients differ by about 1e-12, the robust standard errors by about 5e-12, and the risk differences are equal. Our function uses 30 pattern rows instead of 300,000 single births, so it can fit all 30 million births with very little memory.

## 4. DAG check: are the planned adjustment sets valid?

**What.** Each exposure has a DAG (directed acyclic graph) in `plan/dags/`: a drawing of our assumptions about what causes what. For each exposure we test the planned confounder set with the *back-door criterion*:
- no variable in the set may be caused by the exposure (it would be a mediator);
- the set must block every non-causal path from the exposure to death.

We also test that adding gestational age (GA) makes each set invalid. GA is caused by the exposure and shares unmeasured causes with death, so adjusting for it opens a collider path.

**How.** A path is blocked if the two ends are *d-separated*. We test d-separation with the standard moralization method: keep the exposure, outcome, set and their ancestors; join parents that share a child; drop edge directions; remove the set; the two ends must then be disconnected. Before the test, the exposure's outgoing edges are removed, so that only back-door paths remain.

**Pass rule.** Every planned set is valid, and every set plus GA is invalid. A DAG is an assumption: this check shows that the sets follow from the drawn DAGs, not that the DAGs are true.

In [4]:
import re
from itertools import combinations
from pathlib import Path

DAGS = Path.cwd().parent / "plan" / "dags"


def read_dag(path):
    """Edges of a dagitty file (the simple 'A -> { B C }' and 'A -> B' forms used in plan/dags)."""
    text = re.sub(r"//.*", "", path.read_text())
    edges = set()
    for a, b in re.findall(r"(\w+)\s*->\s*(\{[^}]*\}|\w+)", text):
        for t in b.strip("{}").split():
            edges.add((a, t))
    return edges


def ancestors(edges, nodes):
    out, todo = set(nodes), list(nodes)
    while todo:
        n = todo.pop()
        for a, b in edges:
            if b == n and a not in out:
                out.add(a)
                todo.append(a)
    return out


def d_separated(edges, x, y, z):
    keep = ancestors(edges, {x, y} | set(z))
    sub = {(a, b) for a, b in edges if a in keep and b in keep}
    und = {frozenset(e) for e in sub}
    for child in keep:                                   # moralize: join parents of a common child
        parents = [a for a, b in sub if b == child]
        und |= {frozenset(p) for p in combinations(parents, 2)}
    und = {e for e in und if not (e & set(z))}
    seen, todo = {x}, [x]
    while todo:
        n = todo.pop()
        for e in und:
            if n in e:
                (m,) = e - {n} if len(e) == 2 else (n,)
                if m not in seen:
                    seen.add(m)
                    todo.append(m)
    return y not in seen


def descendants(edges, x):
    out, todo = set(), [x]
    while todo:
        n = todo.pop()
        for a, b in edges:
            if a == n and b not in out:
                out.add(b)
                todo.append(b)
    return out


def backdoor_valid(edges, x, y, z):
    if set(z) & descendants(edges, x):
        return False
    return d_separated({(a, b) for a, b in edges if a != x}, x, y, z)


BASE = ["Age", "Race", "Nativity", "Edu", "Pay", "Parity", "Year"]
PLANNED = {
    "hypertension": ("HTN", BASE + ["Smoke0", "BMI"]),
    "diabetes": ("DM", BASE + ["Smoke0", "BMI"]),
    "bmi": ("BMI", BASE + ["Smoke0"]),
    "smoking": ("Smoke0", BASE),
    "interval": ("IPI", BASE + ["Smoke0", "PriorLoss", "PriorDead", "PrevPTB"]),
}
rows = []
for name, (x, z) in PLANNED.items():
    e = read_dag(DAGS / f"dag_{name}.dagitty")
    rows.append({"exposure": name, "set": ", ".join(z), "valid": backdoor_valid(e, x, "Death", z),
                 "valid_with_GA": backdoor_valid(e, x, "Death", z + ["GA"]),
                 "unadjusted_valid": backdoor_valid(e, x, "Death", [])})
dag_check = pl.DataFrame(rows)
print("DAG CHECK:", "PASS" if dag_check["valid"].all() and not dag_check["valid_with_GA"].any() else "FAIL")
dag_check

DAG CHECK: PASS


exposure,set,valid,valid_with_GA,unadjusted_valid
str,str,bool,bool,bool
"""hypertension""","""Age, Race, Nativity, Edu, Pay,…",true,false,false
"""diabetes""","""Age, Race, Nativity, Edu, Pay,…",true,false,false
"""bmi""","""Age, Race, Nativity, Edu, Pay,…",true,false,false
"""smoking""","""Age, Race, Nativity, Edu, Pay,…",true,false,false
"""interval""","""Age, Race, Nativity, Edu, Pay,…",true,false,false


**Result.** The check prints PASS when every planned set is valid and every set with gestational age added is invalid. `unadjusted_valid = false` shows that each exposure does need adjustment.

**Self-test of the checker.** Below, two cases with a known answer: in a fork `C → X, C → Y` the empty set is invalid and `{C}` is valid; in a collider `X → M ← U → Y` with `X → Y`, adding `M` is invalid.

In [5]:
fork = {("C", "X"), ("C", "Y"), ("X", "Y")}
collider = {("X", "M"), ("U", "M"), ("U", "Y"), ("X", "Y")}
assert not backdoor_valid(fork, "X", "Y", []) and backdoor_valid(fork, "X", "Y", ["C"])
assert backdoor_valid(collider, "X", "Y", []) and not backdoor_valid(collider, "X", "Y", ["M"])
print("checker self-test PASS")

checker self-test PASS


## Summary

- **Main finding so far.** The model code is correct: it reproduces statsmodels exactly on synthetic data. The planned adjustment sets are valid under the DAGs.
- **Open issues.** The Part 2 analysis on real data is not done yet.
- **Next.** Pattern counts per cohort, one confounder set per exposure from the DAGs, then pooled RR, RD, E-values and the pre-set checks.